# Producto Interno: Actividad 1: Películas Similares
**Contexto:** Usamos el dataset `TMDB 5000 Movies` (disponible en EAFIT Interactiva) para encontrar películas similares basándonos en sus descripciones.
**Objetivo:** Encontrar las 3 películas más similares a `"Star Trek Beyond"` utilizando la similitud coseno de sus descripciones textuales (`overview`).
**Plan de Acción:** 1. Vectorización de Texto (TF-IDF). 2. Cálculo de Similitud Coseno.

### 1. Importar librerías y cargar el dataset
En el siguiente bloque importaremos `pandas`, `TfidfVectorizer` y `cosine_similarity`, y cargaremos el archivo CSV para ver cuántas películas contiene.

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Cargar dataset y reemplazar valores nulos en overview por cadena vacía
df = pd.read_csv('tmdb_5000_movies - Copy (1).csv')
df['overview'] = df['overview'].fillna('')
print(f'Películas en dataset: {len(df)}')

Películas en dataset: 4803


### 2. Vectorización TF-IDF
En el siguiente bloque convertiremos los textos de `overview` en vectores numéricos usando `TfidfVectorizer` (con `stop_words='english'` para ignorar palabras comunes). Esto asigna un peso a cada palabra según su importancia.

In [7]:
# Crear vectorizador y transformar los overviews
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['overview'])
print(f'Matriz TF-IDF generada: {tfidf_matrix.shape}')

Matriz TF-IDF generada: (4803, 20978)


### 3. Identificar el índice de la película objetivo
En el siguiente bloque localizaremos la fila correspondiente a `"Star Trek Beyond"` en el dataset, para luego usar su vector como referencia en el cálculo de similitud.

In [8]:
# Encontrar la fila de 'Star Trek Beyond' y extraer su información
target_idx = df[df['title'] == 'Star Trek Beyond'].index[0]
target_title = df.loc[target_idx, 'title']
target_overview = df.loc[target_idx, 'overview']
print(f'Título objetivo: {target_title}')
print(f'Overview: {target_overview}')

Título objetivo: Star Trek Beyond
Overview: The USS Enterprise crew explores the furthest reaches of uncharted space, where they encounter a mysterious new enemy who puts them and everything the Federation stands for to the test.


### 4. Calcular la similitud coseno
En el siguiente bloque calcularemos la similitud coseno entre el vector de `"Star Trek Beyond"` y todos los demás vectores del dataset. Ordenaremos los resultados para ver las películas más cercanas (incluyendo la misma).

In [9]:
# Calcular similitud coseno para todas las películas
cosine_sim = cosine_similarity(tfidf_matrix[target_idx], tfidf_matrix).flatten()
df['similitud'] = cosine_sim

# Ordenar descendente y mostrar las 4 primeras (incluye la misma película)
top4 = df.sort_values(by='similitud', ascending=False).head(4)
print('Resultados ordenados por similitud (incluye la misma película):')
print(top4[['title', 'similitud', 'overview']].to_string(index=False))

Resultados ordenados por similitud (incluye la misma película):
                        title  similitud                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      overview
             Star Trek Beyond   1.000000                                                                                                                                                                                                                                                                                         

### 5. Mostrar las 3 películas más similares (excluyendo la misma)
En el siguiente bloque filtraremos la película objetivo y mostraremos las 3 con mayor similitud coseno, junto con su descripción (`overview`).

In [10]:
# Excluir la película objetivo y tomar las 3 con mayor similitud
top3_excl = df[df['title'] != 'Star Trek Beyond'].sort_values(by='similitud', ascending=False).head(3)
print('Top 3 películas más similares a "Star Trek Beyond":')
for _, row in top3_excl.iterrows():
    print(f"\n- {row['title']} (similitud: {row['similitud']:.4f})")
    print(f"  Overview: {row['overview']}")

Top 3 películas más similares a "Star Trek Beyond":

- Star Trek: First Contact (similitud: 0.1987)
  Overview: The Borg, a relentless race of cyborgs, are on a direct course for Earth. Violating orders to stay away from the battle, Captain Picard and the crew of the newly-commissioned USS Enterprise E pursue the Borg back in time to prevent the invaders from changing Federation history and assimilating the galaxy.

- Star Trek: Insurrection (similitud: 0.1707)
  Overview: When an alien race and factions within Starfleet attempt to take over a planet that has "regenerative" properties, it falls upon Captain Picard and the crew of the Enterprise to defend the planet's people as well as the very ideals upon which the Federation itself was founded.

- Star Trek IV: The Voyage Home (similitud: 0.1418)
  Overview: Fugitives of the Federation for their daring rescue of Spock from the doomed Genesis Planet, Admiral Kirk (William Shatner) and his crew begin their journey home to face justice f